In [ ]:
LCB_SYSTEM_MESSAGE_GENERIC = "You are an expert Python programmer. You will be given a question (problem specification) and will generate a correct Python program that matches the specification and passes all tests."

LCB_FORMATTING_MESSAGE_WITH_STARTER_CODE = "You will use the following starter code to write the solution to the problem and enclose your code within delimiters."

LCB_FORMATTING_WITHOUT_STARTER_CODE = "Read the inputs from stdin solve the problem and write the answer to stdout (do not directly test on the sample inputs). Enclose your code within delimiters as follows. Ensure that when the python program runs, it reads the inputs, runs the algorithm and writes output to STDOUT."

# modified from https://github.com/LiveCodeBench/LiveCodeBench/blob/main/lcb_runner/prompts/code_generation.py
def fetch_live_code_bench_system_prompt(prompt: str, starter_code: str | None = None):
    prompt = LCB_SYSTEM_MESSAGE_GENERIC + "\n\n" + prompt
    if starter_code:
        prompt += f"### Format: {LCB_FORMATTING_MESSAGE_WITH_STARTER_CODE}\n"
        prompt += f"```python\n{starter_code}\n```\n\n"
    else:
        prompt += f"### Format: {LCB_FORMATTING_WITHOUT_STARTER_CODE}\n"
        prompt += "```python\n# YOUR CODE HERE\n```\n\n"
    prompt += "Let's think step by step within <think> </think> tags followed by detailed steps and final code using the provided format with backticks.\n\n"
    return prompt

In [ ]:
import json
from functools import partial
import pandas as pd
from datasets import concatenate_datasets, load_dataset


def filter_tests(tests):
    indexed = [(i, t) for i, t in enumerate(tests)]
    indexed.sort(key=lambda it: (-len(str(it[1].get("input"))), it[0]))
    selected_tests = [t for _, t in indexed[:15]]
    return selected_tests

def preprocess_fn(example, train=True):
    starter_code = example.get("starter_code", "")
    question = fetch_live_code_bench_system_prompt(example["problem"], starter_code if starter_code else None).strip()
    prompt = [{"role": "user", "content": question}]

    tests_raw = example["tests"]
    # Handle different test formats
    if isinstance(tests_raw, str):
        tests = json.loads(tests_raw)
    else:
        tests = tests_raw
    metadata = example.get("metadata", {})

    # Convert TACO format to standard format
    if isinstance(tests, dict) and "inputs" in tests and "outputs" in tests:
        normalized_tests = []
        for input_val, output_val in zip(tests["inputs"], tests["outputs"], strict=False):
            normalized_tests.append({"input": input_val, "output": output_val, "testtype": "stdin_stdout"})
        tests = normalized_tests

    # Ensure tests is always a list
    if not isinstance(tests, list):
        tests = [tests] if tests else []

    for test in tests:
        if test.get("testtype") == "functional" and metadata.get("func_name") is not None:
            test["metadata"] = {"func_name": str(metadata["func_name"])}
        else:
            test["metadata"] = {"func_name": None}

    if train:
        ## Keep max 15 scores based on deepcoder recommondation
        tests = filter_tests(tests)

    reward_model = {"style": "rule", "ground_truth": tests}

    return {
        "prompt": prompt, 
        "reward_model": json.dumps(reward_model), 
        "data_source": "livecodebench",
        "ability": "code",
        "extra_info": {"index": None}
    }

def convert_to_pandas(dataset):
    if hasattr(dataset, "to_pandas") and callable(dataset.to_pandas):
        data_df = dataset.to_pandas()
    else:
        data_df = pd.DataFrame(dataset)
    return data_df

In [ ]:

def _drop_present_columns(ds):
    drop_cols = ["problem", "solutions", "tests", "metadata", "starter_code"]

    present = [c for c in drop_cols if c in ds.column_names]
    return ds.remove_columns(present) if present else ds

def prepare_deepcoder_data(train_size: int = None, test_size: int = None):
    train_dataset = concatenate_datasets([
        load_dataset("agentica-org/DeepCoder-Preview-Dataset", name="primeintellect", split="train"),
        load_dataset("agentica-org/DeepCoder-Preview-Dataset", name="taco", split="train"),
        load_dataset("agentica-org/DeepCoder-Preview-Dataset", name="lcbv5", split="train")
    ])
    test_dataset = concatenate_datasets([
        load_dataset("agentica-org/DeepCoder-Preview-Dataset", name="codeforces", split="test"),
    ])

    if train_size:
        train_dataset = train_dataset.select(range(min(train_size, len(train_dataset))))
    if test_size:
        test_dataset = test_dataset.select(range(min(test_size, len(test_dataset))))

    train_dataset = train_dataset.map(partial(preprocess_fn, train=True), writer_batch_size=10, num_proc=16)
    test_dataset = test_dataset.map(partial(preprocess_fn, train=False), writer_batch_size=10, num_proc=16)

    train_dataset = _drop_present_columns(train_dataset)
    test_dataset = _drop_present_columns(test_dataset)

    return convert_to_pandas(train_dataset), convert_to_pandas(test_dataset)

In [ ]:
train_dataset, test_dataset = prepare_deepcoder_data()

In [ ]:
# Full training set: input to nbs/generate_similar_questions.ipynb and nbs/generate_icl_corpus.ipynb
train_dataset.to_parquet("data/train_deepcoder.parquet", index=False)
train_dataset.sample(10_000).to_parquet("data/train_deepcoder_10K.parquet", index=False)
test_dataset.sample(10).to_parquet("data/test_deepcoder_10.parquet", index=False)

## Testing dataset

In [ ]:

def preprocess_test_fn(example, data_source):
    starter_code = example.get("starter_code", "")
    question = fetch_live_code_bench_system_prompt(example["problem"], starter_code if starter_code else None).strip()
    prompt = [{"role": "user", "content": question}]

    tests_raw = example["tests"]
    # Handle different test formats
    if isinstance(tests_raw, str):
        tests = json.loads(tests_raw)
    else:
        tests = tests_raw
    metadata = example.get("metadata", {})

    # Convert TACO format to standard format
    if isinstance(tests, dict) and "inputs" in tests and "outputs" in tests:
        normalized_tests = []
        for input_val, output_val in zip(tests["inputs"], tests["outputs"], strict=False):
            normalized_tests.append({"input": input_val, "output": output_val, "testtype": "stdin_stdout"})
        tests = normalized_tests

    # Ensure tests is always a list
    if not isinstance(tests, list):
        tests = [tests] if tests else []

    for test in tests:
        if test.get("testtype") == "functional" and metadata.get("func_name") is not None:
            test["metadata"] = {"func_name": str(metadata["func_name"])}
        else:
            test["metadata"] = {"func_name": None}

    reward_model = {"style": "rule", "ground_truth": tests}

    return {
        "prompt": prompt, 
        "reward_model": json.dumps(reward_model), 
        "data_source": data_source,
        "ability": "code",
        "extra_info": {"index": None}
    }

In [ ]:
codeforces_dataset = load_dataset("agentica-org/DeepCoder-Preview-Dataset", name="codeforces", split="test")
lcbv5_dataset = load_dataset("agentica-org/DeepCoder-Preview-Dataset", name="lcbv5", split="test")

In [ ]:
codeforces_dataset = codeforces_dataset.map(partial(preprocess_test_fn, data_source="codeforces"), writer_batch_size=10, num_proc=16)
codeforces_dataset = convert_to_pandas(_drop_present_columns(codeforces_dataset))
lcbv5_dataset = lcbv5_dataset.map(partial(preprocess_test_fn, data_source="livecodebench"), writer_batch_size=10, num_proc=16)
lcbv5_dataset = convert_to_pandas(_drop_present_columns(lcbv5_dataset))


In [ ]:
codeforces_dataset.to_parquet("data/test_codeforces.parquet", index=False)
lcbv5_dataset.to_parquet("data/test_lcbv5.parquet", index=False)